In [ ]:
!pip uninstall -y langchain langchain_community openai
!pip install -q langchain langchain_community langchain-openai faiss-cpu pypdf sentence-transformers

Found existing installation: langchain 1.3.4
Uninstalling langchain-1.3.4:
  Successfully uninstalled langchain-1.3.4
Found existing installation: openai 2.41.0
Uninstalling openai-2.41.0:
  Successfully uninstalled openai-2.41.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.1/127.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into 

In [ ]:
import os
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_2483/226978550.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
# Replace 'your_api_key_here' with your actual Groq API Key, or use secrets manager.
os.environ[""] = ""

In [ ]:
num_pdfs = int(input("How many PDFs would you like to upload? "))

uploaded_files = []
for i in range(num_pdfs):
    print(f"Please upload PDF file {i+1} of {num_pdfs}:")
    upload = files.upload()
    if upload:
        uploaded_files.extend(list(upload.keys()))
    else:
        print("No file uploaded. Skipping this PDF.")

if not uploaded_files:
    raise ValueError("No PDF files were uploaded. Please upload at least one PDF.")

print(f"Successfully uploaded {len(uploaded_files)} PDF(s): {uploaded_files}")

How many PDFs would you like to upload? 2
Please upload PDF file 1 of 2:


Saving file-example_PDF_500_kB.pdf to file-example_PDF_500_kB.pdf
Please upload PDF file 2 of 2:


Saving ex1.txt to ex1.txt
Successfully uploaded 2 PDF(s): ['file-example_PDF_500_kB.pdf', 'ex1.txt']


In [ ]:
all_documents = []
for pdf_file in uploaded_files:
    try:
        loader = PyPDFLoader(pdf_file)
        documents = loader.load()
        all_documents.extend(documents)
        print(f"Loaded {len(documents)} pages from {pdf_file}")
    except Exception as e:
        print(f"Warning: Could not load {pdf_file} as a PDF. Skipping this file. Error: {e}")

if not all_documents:
    raise ValueError("No valid PDF documents were loaded. Please upload at least one valid PDF.")

print("Total Pages Loaded across all PDFs:", len(all_documents))

Loaded 5 pages from file-example_PDF_500_kB.pdf
Total Pages Loaded across all PDFs: 5


In [ ]:
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(all_documents)
print("Total Chunks:", len(docs))

Total Chunks: 12


In [ ]:
# Create embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_2483/3422649668.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.w

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Store in FAISS
vectorstore = FAISS.from_documents(docs, embeddings)
print("Vector DB Created Successfully")

Vector DB Created Successfully


In [ ]:
# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below:
{context}

Question: {question}
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
print("RAG pipeline initialized. You can now ask questions about your documents.\n")

while True:
    user_question = input("Your question (type 'exit' to quit): ")
    if user_question.lower() == 'exit':
        print("Exiting RAG query session.")
        break

    response = rag_chain.invoke(user_question)
    print(f"\nAnswer: {response}\n")

RAG pipeline initialized. You can now ask questions about your documents.

Your question (type 'exit' to quit): vai bhav

Answer: There is no context related to "vai bhav" in the provided text. The text appears to be a passage written in Latin, describing various design elements and layouts, but it does not mention "vai bhav" or provide any information related to it.

Your question (type 'exit' to quit): latine

Answer: The text appears to be written in Latin, with some phrases and sentences repeated. However, there is no specific question being asked, only the word "latine" which is Latin for "in Latin". If you're asking if the text is in Latin, the answer is yes.

Your question (type 'exit' to quit): exit
Exiting RAG query session.
